# 📊 Project 01: 自然語言股票查詢器 — Demo

這個 notebook 展示如何用自然語言查詢股票、計算技術指標、並進行回測分析。

In [ ]:
import sys
sys.path.insert(0, 'projects/01-nl-stock-query/src')

# 1. 自然語言解析
from parser import QueryParser

parser = QueryParser()
result = parser.parse('幫我找最近 RSI 低於 30 的 A 股')
print(f'Intent: {result.intent_type}')
print(f'Indicators: {result.indicators}')
print(f'Scope: {result.stock_scope}')

In [ ]:
# 2. 技術指標計算
import pandas as pd
import numpy as np
from indicators import calculate_rsi, calculate_macd, calculate_bollinger

# 生成模擬股價數據
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=100)
close = 100 + np.cumsum(np.random.randn(100) * 2)
high = close + abs(np.random.randn(100))
low = close - abs(np.random.randn(100))
df = pd.DataFrame({'date': dates, 'open': close + np.random.randn(100)*0.5,
                    'high': high, 'low': low, 'close': close, 'volume': np.random.randint(1000, 5000, 100)})

print('=== RSI ===')
rsi = calculate_rsi(df['close'], period=14)
print(f'Latest RSI: {rsi.iloc[-1]:.2f}')
print(f'RSI < 30 days: {(rsi < 30).sum()}')

print('\n=== MACD ===')
macd_line, signal, histogram = calculate_macd(df['close'])
print(f'MACD: {macd_line.iloc[-1]:.4f}, Signal: {signal.iloc[-1]:.4f}')

print('\n=== Bollinger Bands ===')
upper, middle, lower = calculate_bollinger(df['close'], period=20)
print(f'Upper: {upper.iloc[-1]:.2f}, Middle: {middle.iloc[-1]:.2f}, Lower: {lower.iloc[-1]:.2f}')

In [ ]:
# 3. 策略回測
from backtester import run_backtest

result = run_backtest(
    data=df,
    strategy_type='ma_crossover',
    params={'fast_period': 5, 'slow_period': 20}
)
print(f'Strategy: {result.strategy_name}')
print(f'Total Return: {result.total_return:.2%}')
print(f'Sharpe Ratio: {result.sharpe_ratio:.2f}')
print(f'Max Drawdown: {result.max_drawdown:.2%}')
print(f'Win Rate: {result.win_rate:.2%}')

In [ ]:
# 4. 策略診斷
from diagnostics import diagnose_strategy

diagnosis = diagnose_strategy(result)
print(f'Overall Score: {diagnosis.score}/100')
print(f'Rating: {diagnosis.rating}')
print('\nRecommendations:')
for rec in diagnosis.recommendations:
    print(f'  • {rec}')

## 🎯 Summary

這個項目展示了：
- **NLU 解析**: 自然語言 → 結構化意圖 (Function Calling)
- **技術指標**: RSI, MACD, Bollinger Bands 計算
- **回測引擎**: 4 種策略, Backtrader 整合
- **策略診斷**: 多維度評分 + AI 建議